# IHRAS — 03. Merge: combine the jobs, compute the cross-job statistics, package the evidence

Run after **both** track notebooks have finished, with the same `RUN_ID`.
**Runtime:** CPU with *High RAM* is enough (no GPU units are spent); the bootstrap CIs take a while.

Steps:
1. check that every job has its `.done` marker;
2. merge the job folders with `analysis/merge_jobs.py`; it rejects different samples, conflicting values and diverging files;
3. statistics that need several jobs: `classify`, `fairness`, `external`, `xai_stats`, then the report analyses;
4. figures and comparison with the article, comparison with the published CPU run;
5. evidence: the full package in Drive, plus `evidence_light.zip` to be copied into the repository folder `evidencias/<RUN_ID>/`.

## 1. Configuration

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/pesquisa/ihras2"   # project folder in Google Drive
RUN_ID = "run03"        # MUST be the same in the track A, track B and merge notebooks
TRACK = "M"

REQUIRED_JOBS = ["A0_sampling", "A1_predict_nih", "A2_report", "A3_ablation",
                 "B0_sampling", "B1_explain", "B2_predict_models", "B3_robustness",
                 "B4_official", "B5_timing"]

## 2. Setup

In [ ]:
# ---- Drive, paths, GPU -------------------------------------------------------
import datetime, glob, hashlib, json, os, platform, shlex, shutil, subprocess, sys, time, zipfile
from google.colab import drive
drive.mount("/content/drive")


def utc_now():
    """Current UTC time (ISO 8601), used in every log and provenance record."""
    return datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


SESSION = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN = f"{DRIVE_ROOT}/runs/{RUN_ID}"
JOBS, LOGS = f"{RUN}/jobs", f"{RUN}/logs"           # one folder per job; logs and .done markers
ENV = f"{RUN}/env/track{TRACK}_session_{SESSION}"    # environment record of this session
CODE_SRC, CODE = f"{DRIVE_ROOT}/code", "/content/ihras_code"
DATA_DIR = f"{DRIVE_ROOT}/data/nih"                  # written by 00_data_setup.ipynb
NIH = "/content/nih"                                 # local images and metadata
LOCAL_ZIP = "/content/nih_data.zip"
for d in (JOBS, LOGS, ENV, NIH):
    os.makedirs(d, exist_ok=True)

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
DEVICE = "cuda" if GPU else "cpu"
WORKERS = max(2, (os.cpu_count() or 2) - 1)          # parallel image loaders
print(f"RUN_ID {RUN_ID} | track {TRACK} | session {SESSION} | GPU {GPU} | {os.cpu_count()} vCPUs")

# ---- Code: local copy, checksums, same code in every track ------------------
assert os.path.isdir(CODE_SRC), f"copy the repository folder code/ to {CODE_SRC}"
shutil.rmtree(CODE, ignore_errors=True)
shutil.copytree(CODE_SRC, CODE)


def sha256(path, chunk=1 << 22):
    """SHA-256 of a file, streamed in 4 MB chunks."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


lines = []
for root, dirs, files in os.walk(CODE):
    dirs[:] = sorted(d for d in dirs if not d.startswith(("outputs", "__pycache__", ".")))
    for fn in sorted(files):
        lines.append(f"{sha256(os.path.join(root, fn))}  {os.path.relpath(os.path.join(root, fn), CODE)}")
open(f"{ENV}/code_sha256.txt", "w").write("\n".join(lines) + "\n")
# Only the pipeline code must be identical across tracks and sessions.  Excluded:
# the notebooks in colab/ (Colab rewrites them in Drive when it saves their
# outputs) and hidden files such as .DS_Store, which macOS adds to uploads.
def pipeline(ls):
    keep = []
    for l in ls:
        path = l.split("  ", 1)[1]
        if not path.startswith("colab/") and not os.path.basename(path).startswith("."):
            keep.append(l)
    return keep
PIPELINE = pipeline(lines)
CODE_HASH = hashlib.sha256("\n".join(PIPELINE).encode()).hexdigest()
earlier = sorted((p for p in glob.glob(f"{RUN}/env/*/code_sha256.txt") if not p.startswith(ENV)),
                 key=lambda p: p.split("_session_")[-1])     # earliest session first
if earlier:
    ref = pipeline(open(earlier[0]).read().splitlines())
    changed = sorted(set(ref) ^ set(PIPELINE))
    assert not changed, ("the pipeline code differs from the one this run started with; "
                         "use a new RUN_ID. Differences: %s" % changed[:6])
print("pipeline code hash", CODE_HASH[:16], "| files", len(PIPELINE),
      "| identical to the first session of this run" if earlier else "| first session of this run")

# ---- Dependencies and environment record ------------------------------------
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt",
                "transformers"], check=True)


def capture(cmd, dest):
    """Save the output of a shell command into the environment record."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    open(f"{ENV}/{dest}", "w").write(r.stdout + r.stderr)


capture(f"{sys.executable} -m pip freeze", "pip_freeze.txt")
capture("nvidia-smi", "nvidia-smi.txt")
capture("lscpu", "lscpu.txt")
capture("free -h; df -h /content", "memory_disk.txt")
import numpy, scipy, sklearn, torchxrayvision, transformers
# The versions pinned with "==" in requirements.txt must be the ones loaded: the
# shuffled cross-validation folds depend on the scikit-learn version (run01).
import re as _re
pinned = dict(_re.findall(r"^([A-Za-z0-9_-]+)==([0-9.]+)", open(f"{CODE}/requirements.txt").read(), _re.M))
loaded = {"numpy": numpy.__version__, "scipy": scipy.__version__, "scikit-learn": sklearn.__version__,
          "torchxrayvision": torchxrayvision.__version__}
wrong = {k: (v, loaded[k]) for k, v in pinned.items() if k in loaded and loaded[k] != v}
assert not wrong, ("pinned vs loaded versions differ %s: Runtime > Restart session, "
                   "then run all cells again" % wrong)
json.dump({"python": sys.version, "platform": platform.platform(), "torch": torch.__version__,
           "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(), "gpu": GPU,
           "vcpus": os.cpu_count(), "torchxrayvision": torchxrayvision.__version__,
           "transformers": transformers.__version__, "numpy": numpy.__version__,
           "scipy": scipy.__version__, "scikit-learn": sklearn.__version__,
           "code_hash": CODE_HASH, "track": TRACK, "session_utc": SESSION},
          open(f"{ENV}/versions.json", "w"), indent=1)

# ---- Job runner ---------------------------------------------------------------


def job(name):
    """Output folder of a job (each job writes only to its own folder)."""
    path = f"{JOBS}/{name}"
    os.makedirs(path, exist_ok=True)
    return path


def run(name, args):
    """Run `python -u <args>` inside CODE; stream and log the output to
    logs/<name>.log and write logs/<name>.done on success.  A job with a .done
    marker is skipped, so the notebook can be re-run after a disconnection."""
    done = f"{LOGS}/{name}.done"
    if os.path.exists(done):
        print(f"[{name}] already done: {open(done).read().strip()}")
        return
    cmd = [sys.executable, "-u"] + [str(x) for x in args]
    env = dict(os.environ, CUBLAS_WORKSPACE_CONFIG=":4096:8", PYTHONUNBUFFERED="1",
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")   # less GPU memory fragmentation
    t0 = time.time()
    with open(f"{LOGS}/{name}.log", "a", encoding="utf8") as log:
        log.write(f"\n===== {utc_now()} track={TRACK} session={SESSION} gpu={GPU}\n$ {shlex.join(cmd)}\n")
        p = subprocess.Popen(cmd, cwd=CODE, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end="")
            log.write(line)
        rc = p.wait()
        log.write(f"===== exit={rc} elapsed={time.time() - t0:.0f}s\n")
    if rc:
        raise RuntimeError(f"{name} failed (exit {rc}); see {LOGS}/{name}.log")
    json.dump({"finished_utc": utc_now(), "elapsed_s": round(time.time() - t0), "track": TRACK,
               "session": SESSION, "gpu": GPU, "code_hash": CODE_HASH, "cmd": cmd}, open(done, "w"))


def copy_inputs(dst_job, src_job, files):
    """Give a job the inputs produced by an earlier job (e.g. manifest, predictions)."""
    for fn in files:
        if not os.path.exists(f"{job(dst_job)}/{fn}"):
            shutil.copy2(f"{job(src_job)}/{fn}", f"{job(dst_job)}/{fn}")


def stage(job_name, *args):
    """Arguments of run_all.py for a job folder."""
    return ["run_all.py", "--nih-dir", NIH, "--out", job(job_name), "--workers", WORKERS,
            "--device", DEVICE] + list(args)

## 3. Check the jobs and merge them

In [ ]:
missing = [j for j in REQUIRED_JOBS if not os.path.exists(f"{LOGS}/{j}.done")]
assert not missing, f"jobs not finished yet: {missing}"
manifest = json.load(open(f"{DATA_DIR}/DATA_MANIFEST.json"))
for fn in manifest["metadata_files"]:
    shutil.copy2(f"{DATA_DIR}/meta/{fn}", f"{NIH}/{fn}")   # run_all needs --nih-dir
MERGED = f"{RUN}/merged"
if not os.path.exists(f"{LOGS}/M1_merge.done"):
    shutil.rmtree(MERGED, ignore_errors=True)               # a failed merge leaves nothing behind
run("M1_merge", ["analysis/merge_jobs.py", "--dst", MERGED] + [f"{JOBS}/{j}" for j in REQUIRED_JOBS])

## 4. Cross-job statistics and report analyses

In [ ]:
M = ["run_all.py", "--nih-dir", NIH, "--out", MERGED, "--workers", 2]
run("M2_statistics", M + ["--stages", "classify,fairness,external,xai_stats"])
run("M3_report_fidelity", ["analysis/report_fidelity.py", MERGED])
run("M4_paired_details", ["analysis/paired_details.py", "--out", MERGED])
run("M5_report_hallucination", ["analysis/report_hallucination.py", "--out", MERGED])

## 5. Figures and comparison with the article

In [ ]:
if os.path.exists(f"{LOGS}/M6_figures_compare.done"):
    os.remove(f"{LOGS}/M6_figures_compare.done")          # always redo: cheap and uses the latest keys
run("M6_figures_compare", M + ["--stages", "figures,compare"])
print("".join(open(f"{MERGED}/COMPARISON.md").readlines()[:3]))

## 6. Comparison with the published CPU run (`code/results/cpu_reference/results.json`)

Only the keys of the published run are compared. `time.*` depends on the hardware and is listed separately.

In [ ]:
sys.path.insert(0, CODE)
from ihras.reference_values import tolerance_for
ref = json.load(open(f"{CODE}/results/cpu_reference/results.json"))
new = json.load(open(f"{MERGED}/results.json"))
is_num = lambda v: isinstance(v, (int, float)) and not isinstance(v, bool)
rows, counts = [], {"identical": 0, "within tolerance": 0, "OUT OF TOLERANCE": 0,
                    "time (hardware)": 0, "missing or non-numeric": 0}
for k in sorted(ref):
    a, b = ref[k], new.get(k)
    if not (is_num(a) and is_num(b)):
        if a != b:
            counts["missing or non-numeric"] += 1
            rows.append((k, a, b, None, "missing or non-numeric"))
        continue
    d = abs(b - a)
    status = ("time (hardware)" if k.startswith("time.") else "identical" if d == 0
              else "within tolerance" if d <= tolerance_for(k) else "OUT OF TOLERANCE")
    counts[status] += 1
    if status != "identical":
        rows.append((k, a, b, d, status))
fmt = lambda v: "%.6g" % v if is_num(v) else ("---" if v is None else str(v)[:40])
lines = ["# Colab run vs. published CPU run", "",
         f"RUN_ID {RUN_ID}. " + "; ".join(f"{k}: {v}" for k, v in counts.items()), "",
         "| Key | Published (CPU) | This run | abs. diff | Status |", "|---|---|---|---|---|"]
lines += [f"| {k} | {fmt(a)} | {fmt(b)} | {fmt(d)} | {s} |" for k, a, b, d, s in rows]
open(f"{RUN}/DIFF_vs_cpu_run.md", "w").write("\n".join(lines) + "\n")
print(counts)
for r in rows:
    if r[4] == "OUT OF TOLERANCE":
        print("OUT OF TOLERANCE:", r[0], r[1], "->", r[2])

## 7. Evidence

* `RUN_INFO.json`: jobs, sessions, GPUs, code hash, data checksums, comparison counts.
* `runs/<RUN_ID>.zip`: everything (large files included), kept in Drive and later deposited on Zenodo.
* `evidence_light.zip`: the versionable part, to be copied into the repository folder `evidencias/<RUN_ID>/`.
  Large files are listed in `LARGE_FILES.sha256` with their path in Drive.

In [ ]:
import re
head = open(f"{MERGED}/COMPARISON.md").read()
m = re.search(r"MATCH (\d+) \| MISMATCH (\d+) \| NOT COMPUTED (\d+)", head)
envs = sorted(glob.glob(f"{RUN}/env/*/versions.json"))
info = {"run_id": RUN_ID, "finalized_utc": utc_now(), "code_hash": CODE_HASH,
        "jobs": {os.path.basename(p)[:-5]: json.load(open(p)) for p in sorted(glob.glob(f"{LOGS}/*.done"))},
        "sessions": [json.load(open(p)) for p in envs],
        "data_manifest": manifest,
        "comparison_article": dict(zip(("MATCH", "MISMATCH", "NOT_COMPUTED"), map(int, m.groups()))),
        "diff_vs_cpu_run": counts}
json.dump(info, open(f"{RUN}/RUN_INFO.json", "w"), indent=1)

# Large files: checksums and location only
large = [p for p in glob.glob(f"{RUN}/**/*", recursive=True)
         if os.path.isfile(p) and (p.endswith((".npz", ".jsonl")) or p.endswith("manifest.csv"))]
with open(f"{RUN}/LARGE_FILES.sha256", "w") as f:
    for p in sorted(large):
        f.write(f"{sha256(p)}  {os.path.getsize(p)}  {os.path.relpath(p, DRIVE_ROOT)}\n")

# Full package (Drive / Zenodo)
full_zip = shutil.make_archive(f"/content/{RUN_ID}", "zip", RUN)
shutil.copy(full_zip, f"{DRIVE_ROOT}/runs/{RUN_ID}.zip")

# Light package (repository folder evidencias/<RUN_ID>/)
light = f"/content/evidence_{RUN_ID}"
shutil.rmtree(light, ignore_errors=True)
os.makedirs(f"{light}/tables")
for fn in ("RUN_INFO.json", "DIFF_vs_cpu_run.md", "LARGE_FILES.sha256"):
    shutil.copy2(f"{RUN}/{fn}", light)
for fn in ("results.json", "COMPARISON.md", "merge_provenance.json"):
    shutil.copy2(f"{MERGED}/{fn}", light)
shutil.copytree(f"{MERGED}/figures", f"{light}/figures")
shutil.copytree(LOGS, f"{light}/logs")
shutil.copytree(f"{RUN}/env", f"{light}/env")
for fn in ("xai_per_box.csv", "ablation_per_report.csv", "timing_per_item.csv",
           "report_fidelity_per_report.csv"):
    if os.path.exists(f"{MERGED}/{fn}"):
        shutil.copy2(f"{MERGED}/{fn}", f"{light}/tables/{fn}")
json.dump(manifest, open(f"{light}/DATA_MANIFEST.json", "w"), indent=1)
light_zip = shutil.make_archive(f"/content/evidence_light_{RUN_ID}", "zip", light)
shutil.copy(light_zip, f"{RUN}/evidence_light_{RUN_ID}.zip")
print(json.dumps(info["comparison_article"]), json.dumps(counts))
print("full package :", f"{DRIVE_ROOT}/runs/{RUN_ID}.zip")
print("light package:", f"{RUN}/evidence_light_{RUN_ID}.zip  -> copy into evidencias/{RUN_ID}/")